# Declaration of Originality

**School of Informatics & IT**
<br/>**Diploma in Applied Artificial Intelligence**
<br/>**Machine Learning for Developers (CAI2C08)**
<br/>**AY2026/2027 April Semester**
<br/>**Program Codes**

* Student Name: Marcus Teoh Zhoon Ye (2503577E)


**Declaration of Originality**
* I am the originator of this work, and I have appropriately acknowledged all other original sources used as my references for this work.
* I understand that Plagiarism is the act of taking and using the whole or any part of another person's work, including work generated by AI, and presenting it as my own.
* I understand that Plagiarism is an academic offence and if I am found to have committed or abetted the offence of plagiarism in relation to this submitted work, disciplinary action will be enforced.

**Use of Generative AI tools**

> ⚠️ **TODO — you must write this section yourself, honestly.** The specification says declaring Gen AI use does *not* cost you marks, but an inaccurate declaration is an academic offence. State plainly: which tool, what you used it for (e.g. scaffolding code, debugging, explaining concepts), and confirm that you reviewed and can defend every analytical decision. Delete this quote block once written.


# Libraries

In [1]:
## Import libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# One seed reused everywhere (splits, model init, CV, search) so that every number in
# this notebook is reproducible by the marker on a fresh run.
RANDOM_STATE = 42

# Show all columns; the dataset is only 12 wide so nothing needs truncating.
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

# Consistent plot styling so every figure in the notebook reads as one report.
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titleweight'] = 'bold'

print('pandas', pd.__version__, '| numpy', np.__version__)


pandas 3.0.3 | numpy 2.5.0


# 1. Business Understanding

**Goal:** predict `Item_Outlet_Sales` — the revenue a single product generates at a single outlet — so that a retail chain can plan inventory and evaluate outlet performance before the selling period, not after it.

This is a **regression** problem: the target is a continuous dollar amount, not a class.

**Who uses the prediction, and for what decision**

| Stakeholder | Decision the prediction supports |
|---|---|
| Inventory planner | How many units of this product to send to this outlet |
| Category manager | Which product types earn their shelf space |
| Outlet strategy | Which store formats and locations to open or invest in |

**The cost of being wrong is asymmetric in practice:** over-forecast and you tie up cash in stock that may spoil; under-forecast and you lose the sale outright. We return to this when choosing the evaluation metric in Section 6.

> ✏️ **TODO — expand this into your own paragraph.** Prompts, one line each:
> - Why a *retail chain* cares about per-product-per-store revenue rather than total sales
> - What they currently do instead (hint: a flat average — that is your naive baseline)
> - What a good-enough model unlocks in dollar terms, and what it does **not** solve


# 2. Data Understanding

## 2.1 Load dataset

In [2]:
## Read *.csv file into pandas DataFrame

df = pd.read_csv('data/bigmart.csv')

print(f'Rows: {df.shape[0]:,}   Columns: {df.shape[1]}')
df.head()


Rows: 8,523   Columns: 12


,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


The dataset is the BigMart Sales dataset (Kaggle mirror of the Analytics Vidhya practice problem): **8,523 rows x 12 columns** = 11 candidate features + 1 target. Each row is one *product observed at one outlet*, which is exactly the unit of the business decision above.

**Column meanings**

| Column | Meaning | Level |
|---|---|---|
| `Item_Identifier` | Product code (e.g. FDA15) | Product |
| `Item_Weight` | Product weight | Product |
| `Item_Fat_Content` | Low Fat / Regular | Product |
| `Item_Visibility` | Share of total display area given to this product | Product x Outlet |
| `Item_Type` | Product category (16 levels) | Product |
| `Item_MRP` | Maximum retail price | Product |
| `Outlet_Identifier` | Store code (10 stores) | Outlet |
| `Outlet_Establishment_Year` | Year store opened | Outlet |
| `Outlet_Size` | Small / Medium / High | Outlet |
| `Outlet_Location_Type` | Tier 1 / 2 / 3 city | Outlet |
| `Outlet_Type` | Grocery Store / Supermarket Type1-3 | Outlet |
| **`Item_Outlet_Sales`** | **Target — revenue for this product at this outlet** | Product x Outlet |


## 2.2 Summary Statistics

### 2.2.1 Type of variable for each column

Before any cleaning we establish what each column *is*, because the variable type decides the preprocessing: numeric columns need imputing and scaling, categorical columns need imputing and encoding.

In [3]:
## Understand the type of variable for each column

df.info()

# Cardinality matters as much as dtype here: a column stored as 'object' with 1,559
# distinct values is an identifier, not a usable category, and must be treated differently.
print('\nDistinct values per column:')
print(df.nunique().sort_values(ascending=False).to_string())


<class 'pandas.DataFrame'>
RangeIndex: 8523 entries, 0 to 8522
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Item_Identifier            8523 non-null   str    
 1   Item_Weight                7060 non-null   float64
 2   Item_Fat_Content           8523 non-null   str    
 3   Item_Visibility            8523 non-null   float64
 4   Item_Type                  8523 non-null   str    
 5   Item_MRP                   8523 non-null   float64
 6   Outlet_Identifier          8523 non-null   str    
 7   Outlet_Establishment_Year  8523 non-null   int64  
 8   Outlet_Size                6113 non-null   str    
 9   Outlet_Location_Type       8523 non-null   str    
 10  Outlet_Type                8523 non-null   str    
 11  Item_Outlet_Sales          8523 non-null   float64
dtypes: float64(4), int64(1), str(7)
memory usage: 1.2 MB

Distinct values per column:
Item_Visibility              7880

### 2.2.2 Check for missing data

Missing values decide two things: whether a column is usable at all, and whether the imputation must be **fitted on the training set only** (it must — otherwise test-set information leaks into training).

In [4]:
## Check for missing data

missing = pd.DataFrame({
    'n_missing': df.isna().sum(),
    'pct_missing': (df.isna().mean() * 100).round(1),
})
missing = missing[missing['n_missing'] > 0].sort_values('n_missing', ascending=False)

print('Columns with missing values:')
print(missing.to_string() if len(missing) else 'None')


Columns with missing values:
             n_missing  pct_missing
Outlet_Size       2410         28.3
Item_Weight       1463         17.2


### 2.2.3 Describe data distribution

Summary statistics for the numeric columns, including the target. We are looking for impossible values (a zero where zero cannot occur), and for skew in the target that would affect which error metric is honest.

In [5]:
## Describe data distribution

display(df.describe().T)

# The target deserves its own read: mean vs median tells us about skew, which affects
# both metric choice (Section 6) and whether a log transform is worth testing.
target = df['Item_Outlet_Sales']
print('Target: Item_Outlet_Sales')
print(f'  min     ${target.min():,.2f}')
print(f'  median  ${target.median():,.2f}')
print(f'  mean    ${target.mean():,.2f}')
print(f'  max     ${target.max():,.2f}')
print(f'  skew    {target.skew():.2f}   (0 = symmetric; > 1 = strong right tail)')


,count,mean,std,min,25%,50%,75%,max
Item_Weight,7060.0,12.857645,4.643456,4.555,8.773750,12.600000,16.850000,21.350000
Item_Visibility,8523.0,0.066132,0.051598,0.000,0.026989,0.053931,0.094585,0.328391
Item_MRP,8523.0,140.992782,62.275067,31.290,93.826500,143.012800,185.643700,266.888400
Outlet_Establishment_Year,8523.0,1997.831867,8.371760,1985.000,1987.000000,1999.000000,2004.000000,2009.000000
Item_Outlet_Sales,8523.0,2181.288914,1706.499616,33.290,834.247400,1794.331000,3101.296400,13086.964800


Target: Item_Outlet_Sales
  min     $33.29
  median  $1,794.33
  mean    $2,181.29
  max     $13,086.96
  skew    1.18   (0 = symmetric; > 1 = strong right tail)


### 2.2.4 Categorical levels — label consistency check

*(Added subsection.)* `df.info()` cannot tell us whether a categorical column uses **consistent labels**. If the same real-world category is spelled three different ways, one-hot encoding will create three separate columns for it and split the signal.

In [6]:
## Inspect every level of every categorical column

# pandas 3 stores text columns as the dedicated 'str' dtype, so select on that
# rather than the legacy 'object' alias (which now raises a deprecation warning).
categorical_cols = df.select_dtypes(include='str').columns

for col in categorical_cols:
    # Skip the two identifier columns - listing 1,559 product codes is not informative.
    if df[col].nunique() > 20:
        print(f'{col}: {df[col].nunique()} distinct values (identifier - not listed)\n')
        continue
    print(f'{col}  ({df[col].nunique()} levels)')
    print(df[col].value_counts().to_string())
    print()


Item_Identifier: 1559 distinct values (identifier - not listed)

Item_Fat_Content  (5 levels)
Item_Fat_Content
Low Fat    5089
Regular    2889
LF          316
reg         117
low fat     112

Item_Type  (16 levels)
Item_Type
Fruits and Vegetables    1232
Snack Foods              1200
Household                 910
Frozen Foods              856
Dairy                     682
Canned                    649
Baking Goods              648
Health and Hygiene        520
Soft Drinks               445
Meat                      425
Breads                    251
Hard Drinks               214
Others                    169
Starchy Foods             148
Breakfast                 110
Seafood                    64

Outlet_Identifier  (10 levels)
Outlet_Identifier
OUT027    935
OUT013    932
OUT049    930
OUT046    930
OUT035    930
OUT045    929
OUT018    928
OUT017    926
OUT010    555
OUT019    528

Outlet_Size  (3 levels)
Outlet_Size
Medium    2793
Small     2388
High       932

Outlet_Location_Type  

### 2.2.5 Disguised missing values in `Item_Visibility`

*(Added subsection.)* `Item_Visibility` is the share of display area a product occupies. A product that is stocked and recorded as sold **cannot** occupy 0% of the shelf. Any zero here is therefore a missing value written as a number — the kind that `isna()` will never flag and that silently drags the feature's distribution toward zero.

In [7]:
## Quantify the disguised-missing problem in Item_Visibility

n_zero = (df['Item_Visibility'] == 0).sum()
print(f'Rows with Item_Visibility == 0 : {n_zero:,}  ({n_zero / len(df) * 100:.1f}%)')

# Proof that these are not genuine zeros: the products concerned still recorded sales.
zero_rows = df[df['Item_Visibility'] == 0]
print(f'...of those, rows with sales > 0 : {(zero_rows["Item_Outlet_Sales"] > 0).sum():,}')
print(f'   their median sales           : ${zero_rows["Item_Outlet_Sales"].median():,.2f}')
print(f'   median sales, all other rows : '
      f'${df.loc[df["Item_Visibility"] > 0, "Item_Outlet_Sales"].median():,.2f}')


Rows with Item_Visibility == 0 : 526  (6.2%)
...of those, rows with sales > 0 : 526
   their median sales           : $1,774.02


   median sales, all other rows : $1,794.33


### 2.2.6 Identifier columns and the leakage risk they carry

*(Added subsection.)* `Outlet_Identifier` has only 10 levels, so it is *tempting* to one-hot encode it. The check below shows why that is a trap for this business problem: the 10 stores have very different average sales, so the identifier would let the model memorise each store's historical average instead of learning **why** stores differ (size, type, location, age).

That distinction matters commercially: a model keyed to store IDs cannot answer *'what would a new Tier-2 supermarket sell?'* — which is the outlet-strategy question in Section 1.

In [8]:
## Why Outlet_Identifier is excluded despite having only 10 levels

outlet_profile = (
    df.groupby('Outlet_Identifier')
      .agg(n_rows=('Item_Outlet_Sales', 'size'),
           mean_sales=('Item_Outlet_Sales', 'mean'),
           outlet_type=('Outlet_Type', 'first'),
           outlet_size=('Outlet_Size', lambda s: s.iloc[0] if s.notna().any() else 'missing'),
           location=('Outlet_Location_Type', 'first'))
      .sort_values('mean_sales', ascending=False)
      .round(1)
)
display(outlet_profile)

# If mean sales are almost perfectly explained by Outlet_Type, then Outlet_Type carries
# the same signal in a form that generalises to outlets the model has never seen.
print('Mean sales by Outlet_Type (the generalisable version of the same signal):')
print(df.groupby('Outlet_Type')['Item_Outlet_Sales'].mean().round(1).to_string())


,n_rows,mean_sales,outlet_type,outlet_size,location
Outlet_Identifier,,,,,
OUT027,935,3694.0,Supermarket Type3,Medium,Tier 3
OUT035,930,2438.8,Supermarket Type1,Small,Tier 2
OUT049,930,2348.4,Supermarket Type1,Medium,Tier 1
OUT017,926,2340.7,Supermarket Type1,missing,Tier 2
OUT013,932,2299.0,Supermarket Type1,High,Tier 3
OUT046,930,2277.8,Supermarket Type1,Small,Tier 1
OUT045,929,2192.4,Supermarket Type1,missing,Tier 2
OUT018,928,1995.5,Supermarket Type2,Medium,Tier 3
OUT019,528,340.3,Grocery Store,Small,Tier 1


Mean sales by Outlet_Type (the generalisable version of the same signal):
Outlet_Type
Grocery Store         339.8
Supermarket Type1    2316.2
Supermarket Type2    1995.5
Supermarket Type3    3694.0


### 2.2.7 Is the missing data actually random?

*(Added subsection.)* Standard practice is to impute a missing category with its mode. That is only defensible if the values are missing more or less at random. Before accepting it, we check **where** the missingness sits — because if it is concentrated in a few outlets, the mode is imputing a fact about *other* stores onto stores it does not describe.

The same question for `Item_Weight`: is a product's weight missing at random, or missing for particular products (in which case another row for the *same product* may carry the true weight, and a per-product lookup beats a global median)?

In [9]:
## Where does the missingness actually sit?

# Outlet_Size: count missing rows per outlet. If whole outlets are missing, the
# missingness is structural (an unrecorded store attribute), not random noise.
size_missing = (
    df.assign(size_missing=df['Outlet_Size'].isna())
      .groupby('Outlet_Identifier')
      .agg(n_rows=('size_missing', 'size'),
           n_missing=('size_missing', 'sum'),
           outlet_type=('Outlet_Type', 'first'),
           location=('Outlet_Location_Type', 'first'))
)
print('Outlet_Size missingness by outlet (only outlets with any missing shown):')
print(size_missing[size_missing['n_missing'] > 0].to_string())
print(f"\nOutlets fully missing a size: "
      f"{(size_missing['n_missing'] == size_missing['n_rows']).sum()} of {len(size_missing)}")

# Item_Weight: is the weight recoverable from another row for the SAME product?
weight_by_item = df.groupby('Item_Identifier')['Item_Weight'].nunique(dropna=True)
items_missing = df.loc[df['Item_Weight'].isna(), 'Item_Identifier'].unique()
recoverable = sum(weight_by_item.get(i, 0) > 0 for i in items_missing)
print(f'\nProducts with a missing weight somewhere : {len(items_missing):,}')
print(f'...of which the weight IS recorded on another row : {recoverable:,}')
print(f'Max distinct weights recorded for any one product : {weight_by_item.max()} '
      f'(1 = weight is a stable product attribute)')


Outlet_Size missingness by outlet (only outlets with any missing shown):
                   n_rows  n_missing        outlet_type location
Outlet_Identifier                                               
OUT010                555        555      Grocery Store   Tier 3
OUT017                926        926  Supermarket Type1   Tier 2
OUT045                929        929  Supermarket Type1   Tier 2

Outlets fully missing a size: 3 of 10

Products with a missing weight somewhere : 1,142
...of which the weight IS recorded on another row : 1,138
Max distinct weights recorded for any one product : 1 (1 = weight is a stable product attribute)


### 2.2.8 Data-quality findings and the resulting cleaning plan

> ✏️ **TODO — fill this table in from YOUR outputs above, then write two sentences under it.** Do not copy numbers you have not seen printed. Prompts:
> - For each issue: what is wrong, how big is it (%), and is the fix **deterministic** >   (a rule that needs no data, e.g. relabelling `LF` to `Low Fat`) or **fitted** >   (needs a statistic learned from data, e.g. a median — therefore must go inside the >   pipeline and be fitted on train only)?
> - Which single issue would most distort the model if left alone, and why?

| # | Issue | Column | Size | Fix | Deterministic or fitted? |
|---|---|---|---|---|---|
| 1 | Inconsistent labels | `Item_Fat_Content` | ? | | |
| 2 | Zeros that mean 'missing' | `Item_Visibility` | ? | | |
| 3 | Missing values | `Item_Weight` | ? | | |
| 4 | Missing values | `Outlet_Size` | ? | | |
| 5 | High-cardinality identifiers | `Item_Identifier`, `Outlet_Identifier` | ? | | |
